In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "async-geotiff>=0.4",
#     "lonboard>=0.15.0",
#     "morecantile",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "pillow>=12.1.1",
#     "pystac-client",
#     "sidecar>=0.8.1",
# ]
# ///

# USDA Cropland Data Layer (CDL) from Microsoft Planetary Computer in Lonboard

Visualize a [USDA Cropland Data Layer] (CDL) Cloud-Optimized GeoTIFF tile served from the [Microsoft Planetary Computer] directly in Lonboard — no tile server, fully client-side rendering via [async-geotiff] + [obstore] + [lonboard].

Signing is handled by obstore's built-in [`PlanetaryComputerCredentialProvider`] — no Azure credentials required.

[USDA Cropland Data Layer]: https://www.nass.usda.gov/Research_and_Science/Cropland/SARS1a.php
[Microsoft Planetary Computer]: https://planetarycomputer.microsoft.com/dataset/usda-cdl
[async-geotiff]: https://developmentseed.org/async-geotiff/latest/
[obstore]: https://developmentseed.org/obstore/latest/
[lonboard]: https://developmentseed.org/lonboard/latest/
[`PlanetaryComputerCredentialProvider`]: https://developmentseed.org/obstore/latest/api/auth/planetary-computer/

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and launch with:

```
uvx juv run raster-cog-cdl-pc.ipynb
```

**Optional:** set `PC_SDK_SUBSCRIPTION_KEY` (free signup at [planetarycomputer.microsoft.com](https://planetarycomputer.microsoft.com/account/request)) to bypass anonymous rate limits. Not required.

## Imports

In [ ]:
import io

import numpy as np
import pystac_client
from async_geotiff import GeoTIFF, Tile
from obstore.auth.planetary_computer import PlanetaryComputerCredentialProvider
from obstore.store import AzureStore
from PIL import Image
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard.raster import EncodedImage

## Find a CDL tile via the Planetary Computer STAC API

CDL is hosted as many pre-tiled COGs. We use [pystac-client] to find one — no signing modifier needed here, obstore signs on read.

[pystac-client]: https://pystac-client.readthedocs.io/

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
)

# Iowa corn belt — swap this bbox for anywhere in CONUS.
search = catalog.search(
    collections=["usda-cdl"],
    bbox=[-93.7, 41.5, -93.5, 41.7],
    query={"usda_cdl:type": {"eq": "cropland"}},
    datetime="2021",
)
item = next(search.items())
href = item.assets["cropland"].href
item.id, href

## Open the COG with obstore + the Planetary Computer credential provider

[`PlanetaryComputerCredentialProvider`] takes the full blob URL and produces a fresh SAS token whenever obstore needs one — including auto-refresh as tokens expire.

[`PlanetaryComputerCredentialProvider`]: https://developmentseed.org/obstore/latest/api/auth/planetary-computer/

In [ ]:
# Path within the usda-cdl container, e.g. 'tiles/973905/2021_30m_cdls_…tif'
blob_path = "/".join(href.split("/")[4:])

store = AzureStore(
    credential_provider=PlanetaryComputerCredentialProvider(
        url="https://landcoverdata.blob.core.windows.net/usda-cdl/",
    ),
)
geotiff = await GeoTIFF.open(blob_path, store=store)
geotiff

## Render callback

CDL is a single-band paletted raster — the COG embeds a colormap mapping each class code (corn=1, cotton=2, …) to RGB. We expand each fetched tile into an RGBA PNG entirely in the notebook process.

In [ ]:
cmap_array = geotiff.colormap.as_array()


def render_paletted_tile(tile: Tile) -> EncodedImage:
    arr = tile.array.data[0]  # (H, W) uint8 palette indices

    alpha = np.full(arr.shape, 255, dtype=np.uint8)
    if tile.array.nodata is not None:
        alpha[arr == tile.array.nodata] = 0
    # CDL class 0 = background
    alpha[arr == 0] = 0

    rgba = np.empty((*arr.shape, 4), dtype=np.uint8)
    rgba[..., :3] = cmap_array[arr]
    rgba[..., 3] = alpha

    img = Image.fromarray(rgba, mode="RGBA")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

## Build the layer

In [ ]:
layer = RasterLayer.from_geotiff(geotiff, render_tile=render_paletted_tile)

In [ ]:
sidecar = Sidecar(anchor="split-right")

In [ ]:
m = Map(layer, height=800)

In [ ]:
with sidecar:
    display(m)

## Debug the render callback

Fetch one internal tile from the COG and run the render callback on it directly — useful for inspecting the RGBA output without going through the map.

In [ ]:
num_tiles_x, num_tiles_y = geotiff.tile_count
tile = await geotiff.fetch_tile(num_tiles_x // 2, num_tiles_y // 2)

In [ ]:
render_paletted_tile(tile)